In [1]:
from market_analysis.data.sources import yahoo as yh
from market_analysis.database import reset_database, get_session
from market_analysis.database.models import Security, DailyBar
from market_analysis.database.models.enums import (
    Currency,
    Exchange,
    SecurityType,
    TimeInterval
)

In [2]:
reset_database()

In [3]:
df = yh.download_daily_history(symbol="RELIANCE", exchange="NSE")

In [4]:
with get_session() as session:
    reliance = Security(
        symbol="RELIANCE",
        exchange=Exchange.NSE,
        isin="INE002A01018",
        name="Reliance Industries Limited",
        security_type=SecurityType.STOCK,
        currency=Currency.INR,
    )
    session.add(reliance)
    session.commit()

In [5]:
bars = []

for date, row in df.iterrows():

    bars.append(
        DailyBar(
            security_id=reliance.id,
            date=date.date(),

            open=row["Open"],
            high=row["High"],
            low=row["Low"],
            close=row["Close"],
            adj_close=row["Adj Close"],
            volume=row["Volume"],
        )
    )

In [6]:
session.add_all(bars)
session.commit()

In [7]:
from sqlalchemy import select

stmt = (
    select(DailyBar)
    .where(DailyBar.security_id == reliance.id)
    .order_by(DailyBar.date)
)

bars = session.scalars(stmt).all()

In [8]:
import pandas as pd

df_db = pd.DataFrame(
    {
        "Date": bar.date,
        "Open": bar.open,
        "High": bar.high,
        "Low": bar.low,
        "Close": bar.close,
        "Adj Close": bar.adj_close,
        "Volume": bar.volume,
    }
    for bar in bars
)

df_db.set_index("Date", inplace=True)

In [18]:
from market_analysis.charts.candlestick import plot_candlestick

In [20]:
plot_candlestick(df_db, show_volume=True)

In [15]:
df.head(10)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-06-16,1328.800049,1328.800049,1333.400024,1306.400024,1313.400024,18513223
2026-06-17,1332.699951,1332.699951,1334.000000,1317.000000,1333.000000,10029170
2026-06-18,1328.099976,1328.099976,1333.900024,1322.000000,1330.000000,15494549
2026-06-19,1309.500000,1309.500000,1338.199951,1305.300049,1328.000000,24887034
2026-06-22,1326.500000,1326.500000,1344.900024,1314.099976,1316.699951,12931213
2026-06-23,1309.500000,1309.500000,1333.000000,1304.000000,1328.900024,15400184
2026-06-24,1313.599976,1313.599976,1322.000000,1297.500000,1305.699951,11030917
2026-06-25,1318.099976,1318.099976,1328.000000,1314.599976,1318.000000,12694362
2026-06-26,1318.099976,1318.099976,1318.099976,1318.099976,1318.099976,0


In [17]:
df_db.head(10)

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2026-06-16,1313.400024,1333.400024,1306.400024,1328.800049,1328.800049,18513223.0
2026-06-17,1333.000000,1334.000000,1317.000000,1332.699951,1332.699951,10029170.0
2026-06-18,1330.000000,1333.900024,1322.000000,1328.099976,1328.099976,15494549.0
2026-06-19,1328.000000,1338.199951,1305.300049,1309.500000,1309.500000,24887034.0
2026-06-22,1316.699951,1344.900024,1314.099976,1326.500000,1326.500000,12931213.0
2026-06-23,1328.900024,1333.000000,1304.000000,1309.500000,1309.500000,15400184.0
2026-06-24,1305.699951,1322.000000,1297.500000,1313.599976,1313.599976,11030917.0
2026-06-25,1318.000000,1328.000000,1314.599976,1318.099976,1318.099976,12694362.0
2026-06-26,1318.099976,1318.099976,1318.099976,1318.099976,1318.099976,0.0
